In [6]:
!pip install nltk

In [7]:
import numpy as np
import pandas as pd
import regex as re
import nltk
from google.colab import drive
from transformers import pipeline
import torch
device = 0 if torch.cuda.is_available() else -1
device

-1

In [8]:
drive.mount('/content/drive')
reviews = pd.read_csv('/content/drive/MyDrive/Sephora/reviews_cleaned.csv')
reviews.head()

Mounted at /content/drive


/tmp/ipykernel_465/1059886061.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv('/content/drive/MyDrive/Sephora/reviews_cleaned.csv')


,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd,help_per
0,1945004256,5,Recommended,0.000000,2,2,0,2022-12-10,I absolutely L-O-V-E this oil. I have acne pro...,lightMedium,green,combination,Unknown,P379064,Lotus Balancing & Hydrating Natural Face Treat...,Clarins,65.0,0.0
1,5478482359,3,Recommended,0.333333,3,2,1,2021-12-17,I gave this 3 stars because it give me tiny li...,mediumTan,brown,oily,black,P379064,Lotus Balancing & Hydrating Natural Face Treat...,Clarins,65.0,0.0
2,29002209922,5,Recommended,1.000000,2,0,2,2021-06-07,Works well as soon as I wash my face and pat d...,lightMedium,brown,dry,black,P379064,Lotus Balancing & Hydrating Natural Face Treat...,Clarins,65.0,100.0
3,7391078463,5,Recommended,1.000000,2,0,2,2021-05-21,"this oil helped with hydration and breakouts, ...",lightMedium,brown,combination,blonde,P379064,Lotus Balancing & Hydrating Natural Face Treat...,Clarins,65.0,100.0
4,1766313888,5,Recommended,1.000000,13,0,13,2021-03-29,This is my first product review ever so that s...,mediumTan,brown,combination,black,P379064,Lotus Balancing & Hydrating Natural Face Treat...,Clarins,65.0,100.0


In [9]:
sent_pipe = pipeline(
    "sentiment-analysis",
    model = 'distilbert/distilbert-base-uncased-finetuned-sst-2-english' ,
    truncation = True ,
    device = device ,
  )

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [10]:
def sent_analysis(texts):
  results = sent_pipe(texts)
  scores = []

  for res in results:
    if res['label'] == 'POSITIVE':
      scores.append(res['score'])
    else:
      scores.append(1 - res['score'])

  return scores

# **Stratified Sampling of products reviews**


*   NLP Preprocssing
*   Sentiment Analysis



In [11]:
unique_prods = reviews['product_id'].nunique()
unique_prods

2351

In [12]:
sample_reviews = reviews.groupby('product_id', group_keys=False).apply(
    lambda x: x.sample(frac=0.10, random_state=42) if len(x) >= 1 else x
).reset_index(drop=True)
sample_reviews.shape

/tmp/ipykernel_465/1052078508.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sample_reviews = reviews.groupby('product_id', group_keys=False).apply(


(109292, 18)

In [13]:
# Lower & Punctuation removal
sample_reviews['review_text'] = sample_reviews['review_text'].apply(lambda x: x.lower())
sample_reviews['review_text'] = sample_reviews['review_text'].apply(
    lambda x :re.sub(r"[^\w\s']", '', x)
)
sample_reviews.head()

,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd,help_per
0,1351262898,5,Not Recommended,0.000000,0,0,0,2010-03-06,i cant believe that i have seen results in les...,light,Unknown,normal,Unknown,P107306,Renewing Eye Cream,Murad,89.0,0.0
1,7152044287,5,Recommended,0.333333,3,2,1,2021-12-20,cute packaging im really impressed with the re...,lightMedium,brown,combination,brown,P107306,Renewing Eye Cream,Murad,89.0,0.0
2,6742580347,5,Recommended,0.800000,20,4,16,2017-07-31,never before have i seen such positive results...,Unknown,Unknown,normal,Unknown,P107306,Renewing Eye Cream,Murad,89.0,100.0
3,1533231934,4,Recommended,0.833333,6,1,5,2010-04-06,ive been using murads essentialc eye cream spf...,fair,Unknown,oily,Unknown,P107306,Renewing Eye Cream,Murad,89.0,100.0
4,1919772119,4,Not Recommended,0.000000,0,0,0,2015-05-03,i dont think ive noticed a difference in my da...,Unknown,Unknown,Unknown,Unknown,P107306,Renewing Eye Cream,Murad,89.0,0.0


In [14]:
# Replacing contractions
contractions = {
    "can't": "cannot", "won't": "will not", "don't": "do not",
    "didn't": "did not", "isn't": "is not", "aren't": "are not",
    "wasn't": "was not", "weren't": "were not", "haven't": "have not",
    "hasn't": "has not", "hadn't": "had not", "wouldn't": "would not",
    "couldn't": "could not", "shouldn't": "should not", "i'm": "i am",
    "i've": "i have", "i'll": "i will", "i'd": "i would",
    "it's": "it is", "he's": "he is", "she's": "she is",
    "they're": "they are", "we're": "we are", "you're": "you are",
    "that's": "that is", "there's": "there is", "what's": "what is",
    "let's": "let us", "who's": "who is", "here's": "here is",
}
sample_reviews['review_text'] = sample_reviews['review_text'].apply(
    lambda x: ' '.join([contractions[word] if word in contractions else word for word in x.split()])
)
sample_reviews.head()

,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd,help_per
0,1351262898,5,Not Recommended,0.000000,0,0,0,2010-03-06,i cant believe that i have seen results in les...,light,Unknown,normal,Unknown,P107306,Renewing Eye Cream,Murad,89.0,0.0
1,7152044287,5,Recommended,0.333333,3,2,1,2021-12-20,cute packaging im really impressed with the re...,lightMedium,brown,combination,brown,P107306,Renewing Eye Cream,Murad,89.0,0.0
2,6742580347,5,Recommended,0.800000,20,4,16,2017-07-31,never before have i seen such positive results...,Unknown,Unknown,normal,Unknown,P107306,Renewing Eye Cream,Murad,89.0,100.0
3,1533231934,4,Recommended,0.833333,6,1,5,2010-04-06,ive been using murads essentialc eye cream spf...,fair,Unknown,oily,Unknown,P107306,Renewing Eye Cream,Murad,89.0,100.0
4,1919772119,4,Not Recommended,0.000000,0,0,0,2015-05-03,i dont think ive noticed a difference in my da...,Unknown,Unknown,Unknown,Unknown,P107306,Renewing Eye Cream,Murad,89.0,0.0


In [15]:
sample_reviews['review_text'].value_counts()

,count
review_text,
i received this in a sample i have alot of acne scars this serum works it doesnt happen overnight so have patience i have noticed my scars are lightening and its so nice i have tried everyrhing from drug store brands to presrciption medication wirh no results this works go kate,6
this exfoliant will change your face ive used it for years and am so happy its at sephora now you dont need to use very much and i found it worked best only once or twice a week max it helped to improve my skin tone and glow its quite expensive but you only need about dime size,4
makes your face soft and it does not dry your face a little goes a long way too,4
i really enjoy this product the facial cream is extremely hydrating and absorbs into the skin quickly causing no issues with applying before makeup i have very dry skin and this moisturizer lasts the entire day and is great for mixing serums with at night i will use this product again,4
i really love this cleansing oil ive always had dry skin and after the first time my face felt so hydrated and soft it did not feel oily after washing off either i have been using this for about 10 days and my face has not felt this good in years i highly recommend this,4
...,...
super moisturizing i can see this lasting a long time,1
this is a very moisturizing eye cream that stays where you put it i can wear this under my eye makeup without it causing my makeup to slide i am also very prone to milia and this product has not given me any issues its also great on smile lines,1
i like kiehls but this one is a miss for me i have sensitive skin and this thing burns so bad with even the smallest amount on wish i could love it,1


In [16]:
sample_reviews['product_id'].nunique()

2207

In [17]:
rev_texts = sample_reviews['review_text'].tolist()
text_score = []
batch_size = 128
for i in range(0, len(rev_texts) , batch_size ):
  text_score.extend(sent_analysis(rev_texts[i:i+batch_size]))

sample_reviews['sentiment_analysis'] = text_score
sample_reviews.head()

KeyboardInterrupt: 

In [18]:
reviews

,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd,help_per
0,1945004256,5,Recommended,0.000000,2,2,0,2022-12-10,I absolutely L-O-V-E this oil. I have acne pro...,lightMedium,green,combination,Unknown,P379064,Lotus Balancing & Hydrating Natural Face Treat...,Clarins,65.0,0.0
1,5478482359,3,Recommended,0.333333,3,2,1,2021-12-17,I gave this 3 stars because it give me tiny li...,mediumTan,brown,oily,black,P379064,Lotus Balancing & Hydrating Natural Face Treat...,Clarins,65.0,0.0
2,29002209922,5,Recommended,1.000000,2,0,2,2021-06-07,Works well as soon as I wash my face and pat d...,lightMedium,brown,dry,black,P379064,Lotus Balancing & Hydrating Natural Face Treat...,Clarins,65.0,100.0
3,7391078463,5,Recommended,1.000000,2,0,2,2021-05-21,"this oil helped with hydration and breakouts, ...",lightMedium,brown,combination,blonde,P379064,Lotus Balancing & Hydrating Natural Face Treat...,Clarins,65.0,100.0
4,1766313888,5,Recommended,1.000000,13,0,13,2021-03-29,This is my first product review ever so that s...,mediumTan,brown,combination,black,P379064,Lotus Balancing & Hydrating Natural Face Treat...,Clarins,65.0,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1092962,9109189891,5,Recommended,0.000000,0,0,0,2022-04-29,I LOVE this stuff! It works amazingly well at ...,Unknown,Unknown,Unknown,Unknown,P500101,Hydro Ungrip Makeup Remover + Cleansing Water,MILK MAKEUP,32.0,0.0
1092963,8115925555,5,Recommended,0.000000,0,0,0,2022-04-29,love this micellar water from Milk! it removes...,medium,brown,oily,red,P500101,Hydro Ungrip Makeup Remover + Cleansing Water,MILK MAKEUP,32.0,0.0
1092964,10723607564,5,Recommended,0.000000,0,0,0,2022-04-29,I loveeeeee Milk!!! Just discovered this brand...,mediumTan,brown,dry,black,P500101,Hydro Ungrip Makeup Remover + Cleansing Water,MILK MAKEUP,32.0,0.0
1092965,5953458355,5,Recommended,0.000000,0,0,0,2022-04-29,"My new favorite makeup remover. First of all, ...",light,blue,combination,blonde,P500101,Hydro Ungrip Makeup Remover + Cleansing Water,MILK MAKEUP,32.0,0.0


In [19]:
sample_reviews

,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd,help_per
0,1351262898,5,Not Recommended,0.000000,0,0,0,2010-03-06,i cant believe that i have seen results in les...,light,Unknown,normal,Unknown,P107306,Renewing Eye Cream,Murad,89.0,0.0
1,7152044287,5,Recommended,0.333333,3,2,1,2021-12-20,cute packaging im really impressed with the re...,lightMedium,brown,combination,brown,P107306,Renewing Eye Cream,Murad,89.0,0.0
2,6742580347,5,Recommended,0.800000,20,4,16,2017-07-31,never before have i seen such positive results...,Unknown,Unknown,normal,Unknown,P107306,Renewing Eye Cream,Murad,89.0,100.0
3,1533231934,4,Recommended,0.833333,6,1,5,2010-04-06,ive been using murads essentialc eye cream spf...,fair,Unknown,oily,Unknown,P107306,Renewing Eye Cream,Murad,89.0,100.0
4,1919772119,4,Not Recommended,0.000000,0,0,0,2015-05-03,i dont think ive noticed a difference in my da...,Unknown,Unknown,Unknown,Unknown,P107306,Renewing Eye Cream,Murad,89.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109287,1892552776,5,Not Recommended,0.000000,0,0,0,2011-08-11,i bought the value gift set for murads sun dam...,lightMedium,Unknown,normal,Unknown,P9941,Essential-C Cleanser,Murad,44.0,0.0
109288,6649985255,5,Recommended,1.000000,3,0,3,2019-12-27,actually leaves face feeling clean without ove...,rich,brown,combination,black,P9941,Essential-C Cleanser,Murad,44.0,100.0
109289,1770780350,4,Not Recommended,0.000000,0,0,0,2009-07-29,i believe murad essentialc cleanser is the bes...,light,Unknown,combination,Unknown,P9941,Essential-C Cleanser,Murad,44.0,0.0
109290,23940992103,3,Not Recommended,0.600000,5,2,3,2021-04-28,i dont dislike this cleanser and i dont hate i...,deep,brown,oily,black,P9941,Essential-C Cleanser,Murad,44.0,100.0
